In [1]:
# GLOBAL LIBRAIRIES
import importlib, sys, os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

# INITIALIZATION
spark = (
    SparkSession.builder
        .appName("etl_punctuality_data")
        # latest GA driver as of mid-2025 – change version if a newer one appears
        .getOrCreate()
)
# MANDATORY !!!!
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/19 13:40:20 WARN Utils: Your hostname, BEBEL-PC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/19 13:40:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 13:40:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [14]:
punctuality_data_df = spark.read.csv("data/punctuality/202501.csv", header=True, sep=",")

In [3]:
punctuality_data_df.printSchema()
punctuality_data_df.show(5)

root
 |-- DATDEP: string (nullable = true)
 |-- TRAIN_NO: string (nullable = true)
 |-- RELATION: string (nullable = true)
 |-- TRAIN_SERV: string (nullable = true)
 |-- PTCAR_NO: string (nullable = true)
 |-- THOP1_COD: string (nullable = true)
 |-- LINE_NO_DEP: string (nullable = true)
 |-- REAL_TIME_ARR: string (nullable = true)
 |-- REAL_TIME_DEP: string (nullable = true)
 |-- PLANNED_TIME_ARR: string (nullable = true)
 |-- PLANNED_TIME_DEP: string (nullable = true)
 |-- DELAY_ARR: string (nullable = true)
 |-- DELAY_DEP: string (nullable = true)
 |-- CIRC_TYP: string (nullable = true)
 |-- RELATION_DIRECTION: string (nullable = true)
 |-- PTCAR_LG_NM_NL: string (nullable = true)
 |-- LINE_NO_ARR: string (nullable = true)
 |-- PLANNED_DATE_ARR: string (nullable = true)
 |-- PLANNED_DATE_DEP: string (nullable = true)
 |-- REAL_DATE_ARR: string (nullable = true)
 |-- REAL_DATE_DEP: string (nullable = true)

+---------+--------+--------+----------+--------+---------+-----------+------

## Data validation

### Unique values

In [4]:
unique_counts = []
for column in punctuality_data_df.columns :
    unique_count : int = punctuality_data_df.agg(F.count_distinct(column).alias('count')).collect()[0]['count']
    unique_counts.append((column, unique_count))
for col_name, unique_count in unique_counts :
    print(f"Column {col_name} has {unique_count} unique values.")

Column DATDEP has 31 unique values.
Column TRAIN_NO has 4771 unique values.
Column RELATION has 111 unique values.
Column TRAIN_SERV has 3 unique values.
Column PTCAR_NO has 628 unique values.
Column THOP1_COD has 9 unique values.
Column LINE_NO_DEP has 148 unique values.
Column REAL_TIME_ARR has 74521 unique values.
Column REAL_TIME_DEP has 74487 unique values.
Column PLANNED_TIME_ARR has 7586 unique values.
Column PLANNED_TIME_DEP has 5398 unique values.
Column DELAY_ARR has 5509 unique values.
Column DELAY_DEP has 5419 unique values.
Column CIRC_TYP has 1 unique values.
Column RELATION_DIRECTION has 276 unique values.
Column PTCAR_LG_NM_NL has 628 unique values.
Column LINE_NO_ARR has 149 unique values.
Column PLANNED_DATE_ARR has 32 unique values.
Column PLANNED_DATE_DEP has 32 unique values.
Column REAL_DATE_ARR has 32 unique values.
Column REAL_DATE_DEP has 32 unique values.


### Null values

In [5]:
null_counts = []
for column in punctuality_data_df.columns:
    null_count = punctuality_data_df.filter(F.col(column).isNull()).count()
    null_counts.append((column, null_count))
for col_name, null_count in null_counts:
    print(f"Column '{col_name}' has {null_count} null values")

Column 'DATDEP' has 0 null values
Column 'TRAIN_NO' has 0 null values
Column 'RELATION' has 0 null values
Column 'TRAIN_SERV' has 0 null values
Column 'PTCAR_NO' has 0 null values
Column 'THOP1_COD' has 195513 null values
Column 'LINE_NO_DEP' has 95790 null values
Column 'REAL_TIME_ARR' has 95944 null values
Column 'REAL_TIME_DEP' has 95779 null values
Column 'PLANNED_TIME_ARR' has 95944 null values
Column 'PLANNED_TIME_DEP' has 95779 null values
Column 'DELAY_ARR' has 95940 null values
Column 'DELAY_DEP' has 95773 null values
Column 'CIRC_TYP' has 0 null values
Column 'RELATION_DIRECTION' has 99162 null values
Column 'PTCAR_LG_NM_NL' has 0 null values
Column 'LINE_NO_ARR' has 95960 null values
Column 'PLANNED_DATE_ARR' has 95944 null values
Column 'PLANNED_DATE_DEP' has 95779 null values
Column 'REAL_DATE_ARR' has 95944 null values
Column 'REAL_DATE_DEP' has 95779 null values


### Empty values

In [6]:
empty_counts = []
string_columns = [f.name for f in punctuality_data_df.schema.fields if isinstance(f.dataType, T.StringType)]
for column in string_columns:
    empty_count = punctuality_data_df.filter(F.col(column) == "").count()
    empty_counts.append((column, empty_count))
for col_name, empty_count in empty_counts:
    print(f"Column '{col_name}' has {empty_count} empty values")

Column 'DATDEP' has 0 empty values
Column 'TRAIN_NO' has 0 empty values
Column 'RELATION' has 0 empty values
Column 'TRAIN_SERV' has 0 empty values
Column 'PTCAR_NO' has 0 empty values
Column 'THOP1_COD' has 0 empty values
Column 'LINE_NO_DEP' has 0 empty values
Column 'REAL_TIME_ARR' has 0 empty values
Column 'REAL_TIME_DEP' has 0 empty values
Column 'PLANNED_TIME_ARR' has 0 empty values
Column 'PLANNED_TIME_DEP' has 0 empty values
Column 'DELAY_ARR' has 0 empty values
Column 'DELAY_DEP' has 0 empty values
Column 'CIRC_TYP' has 0 empty values
Column 'RELATION_DIRECTION' has 0 empty values
Column 'PTCAR_LG_NM_NL' has 0 empty values
Column 'LINE_NO_ARR' has 0 empty values
Column 'PLANNED_DATE_ARR' has 0 empty values
Column 'PLANNED_DATE_DEP' has 0 empty values
Column 'REAL_DATE_ARR' has 0 empty values
Column 'REAL_DATE_DEP' has 0 empty values


### Duplicate values

In [7]:
duplicates = punctuality_data_df.groupBy(punctuality_data_df.columns).count().filter(F.col("count") > 1)
duplicate_count = duplicates.count()
if duplicate_count > 0:
    print(f"There are {duplicate_count} duplicate lines in the data")
else:
    print("No duplicate lines found in the data")

No duplicate lines found in the data


## Data transformation

### Deletion of unecessary columns

In [15]:
punctuality_data_df = (punctuality_data_df
    .drop("PTCAR_LG_NM_NL", "REAL_TIME_ARR", "REAL_TIME_DEP", "DATDEP")
    .withColumnRenamed("PTCAR_NO", "STOPPING_PLACE_ID")
)
punctuality_data_df.show(5)

+--------+--------+----------+-----------------+---------+-----------+----------------+----------------+---------+---------+--------+--------------------+-----------+----------------+----------------+-------------+-------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|PLANNED_TIME_ARR|PLANNED_TIME_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|PLANNED_DATE_ARR|PLANNED_DATE_DEP|REAL_DATE_ARR|REAL_DATE_DEP|
+--------+--------+----------+-----------------+---------+-----------+----------------+----------------+---------+---------+--------+--------------------+-----------+----------------+----------------+-------------+-------------+
|    3117|   IC 31| SNCB/NMBS|              259|     NULL|        124|            NULL|        17:05:00|     NULL|       40|       1|IC 31: CHARLEROI-...|       NULL|            NULL|       01JAN2025|         NULL|    01JAN2025|
|    3117|   IC 31| SNCB/NMBS|              791|        =|        124|        17:10:

## Merge time and date in one column 

In [16]:
punctuality_data_df = punctuality_data_df.withColumn("PLANNED_DATETIME_ARR", 
    F.concat_ws(" ", 
		F.date_format(F.to_date(F.col("PLANNED_DATE_ARR"), "ddMMMyyyy"), "dd-MM-yyyy").cast("string"), 
		F.col("PLANNED_TIME_ARR")
    )
)
punctuality_data_df = punctuality_data_df.withColumn("PLANNED_DATETIME_DEP", 
    F.concat_ws(" ", 
		F.date_format(F.to_date(F.col("PLANNED_DATE_DEP"), "ddMMMyyyy"), "dd-MM-yyyy").cast("string"), 
		F.col("PLANNED_TIME_DEP")
    )
)
punctuality_data_df = punctuality_data_df.drop("PLANNED_DATE_ARR", "PLANNED_DATE_DEP", "PLANNED_TIME_DEP", "PLANNED_TIME_ARR")
punctuality_data_df.show(5)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|
+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|    3117|   IC 31| SNCB/NMBS|              259|     NULL|        124|     NULL|       40|       1|IC 31: CHARLEROI-...|       NULL|         NULL|    01JAN2025|                    | 01-01-2025 17:05:00|
|    3117|   IC 31| SNCB/NMBS|              791|        =|        124|       -9|       27|       1|IC 31: CHARLEROI-...|        124|    01JAN2025|    01JAN2025| 01-01-2025 17:10:00| 01-01-

## Changing empty values into Null values

In [17]:
empty_counts = []
string_columns = [f.name for f in punctuality_data_df.schema.fields if isinstance(f.dataType, T.StringType)]

for column in string_columns:
    empty_count = punctuality_data_df.filter(F.col(column) == "").count()
    empty_counts.append((column, empty_count))
for col_name, empty_count in empty_counts:
    print(f"Column '{col_name}' has {empty_count} empty values")

Column 'TRAIN_NO' has 0 empty values
Column 'RELATION' has 0 empty values
Column 'TRAIN_SERV' has 0 empty values
Column 'STOPPING_PLACE_ID' has 0 empty values
Column 'THOP1_COD' has 0 empty values
Column 'LINE_NO_DEP' has 0 empty values
Column 'DELAY_ARR' has 0 empty values
Column 'DELAY_DEP' has 0 empty values
Column 'CIRC_TYP' has 0 empty values
Column 'RELATION_DIRECTION' has 0 empty values
Column 'LINE_NO_ARR' has 0 empty values
Column 'REAL_DATE_ARR' has 0 empty values
Column 'REAL_DATE_DEP' has 0 empty values
Column 'PLANNED_DATETIME_ARR' has 95944 empty values
Column 'PLANNED_DATETIME_DEP' has 95779 empty values


In [18]:
punctuality_data_df = punctuality_data_df
for column in string_columns:
    punctuality_data_df = punctuality_data_df.withColumn(column, F.when(F.col(column) == "", None).otherwise(F.col(column)))
punctuality_data_df.show(5)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|
+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|    3117|   IC 31| SNCB/NMBS|              259|     NULL|        124|     NULL|       40|       1|IC 31: CHARLEROI-...|       NULL|         NULL|    01JAN2025|                NULL| 01-01-2025 17:05:00|
|    3117|   IC 31| SNCB/NMBS|              791|        =|        124|       -9|       27|       1|IC 31: CHARLEROI-...|        124|    01JAN2025|    01JAN2025| 01-01-2025 17:10:00| 01-01-

## Convert to right data types

In [19]:
punctuality_data_df = punctuality_data_df.withColumn("TRAIN_NO", F.col("TRAIN_NO").cast("int"))\
.withColumn("STOPPING_PLACE_ID", F.col("STOPPING_PLACE_ID").cast("int"))\
.withColumn("LINE_NO_DEP", F.col("LINE_NO_DEP").cast("string"))\
.withColumn("DELAY_ARR", F.col("DELAY_ARR").cast("int"))\
.withColumn("DELAY_DEP", F.col("DELAY_DEP").cast("int"))\
.withColumn("CIRC_TYP", F.col("CIRC_TYP").cast("int"))\
.withColumn("CIRC_TYP", F.col("CIRC_TYP").cast("int"))\
.withColumn("LINE_NO_ARR", F.col("LINE_NO_ARR").cast("string"))\
.withColumn("PLANNED_DATETIME_ARR", 
    F.to_timestamp(F.col("PLANNED_DATETIME_ARR"), "dd-MM-yyyy H:mm:ss"))\
.withColumn("PLANNED_DATETIME_DEP", 
    F.to_timestamp(F.col("PLANNED_DATETIME_DEP"), "dd-MM-yyyy H:mm:ss"))\
.withColumn("REAL_DATE_ARR", F.date_format(F.to_date(F.col("REAL_DATE_ARR"), "ddMMMyyyy"), "dd-MM-yyyy"))\
.withColumn("REAL_DATE_DEP", F.date_format(F.to_date(F.col("REAL_DATE_DEP"), "ddMMMyyyy"), "dd-MM-yyyy"))\
.withColumn("TRAIN_SERV", F.col("TRAIN_SERV").cast("string"))\
.withColumn("RELATION_DIRECTION", F.col("RELATION_DIRECTION").cast("string"))
punctuality_data_df.printSchema()
punctuality_data_df.show(5)

root
 |-- TRAIN_NO: integer (nullable = true)
 |-- RELATION: string (nullable = true)
 |-- TRAIN_SERV: string (nullable = true)
 |-- STOPPING_PLACE_ID: integer (nullable = true)
 |-- THOP1_COD: string (nullable = true)
 |-- LINE_NO_DEP: string (nullable = true)
 |-- DELAY_ARR: integer (nullable = true)
 |-- DELAY_DEP: integer (nullable = true)
 |-- CIRC_TYP: integer (nullable = true)
 |-- RELATION_DIRECTION: string (nullable = true)
 |-- LINE_NO_ARR: string (nullable = true)
 |-- REAL_DATE_ARR: string (nullable = true)
 |-- REAL_DATE_DEP: string (nullable = true)
 |-- PLANNED_DATETIME_ARR: timestamp (nullable = true)
 |-- PLANNED_DATETIME_DEP: timestamp (nullable = true)

+--------+--------+----------+-----------------+---------+-----------+---------+---------+--------+--------------------+-----------+-------------+-------------+--------------------+--------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|THOP1_COD|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|CIRC_TYP|  RELATION_DIRECT

In [20]:
punctuality_data_df.coalesce(1).write.mode("overwrite").option("header", True).csv("202501", sep=";")